# Running SELDON Inference Using Hyrax

This notebook's goal is to teach you how to run data through the [SELDON](https://github.com/skai-institute/seldon) foundational model developed by the SkAI Institute using [Hyrax](https://github.com/lincc-frameworks/hyrax), the low code framework for machine learning in astronomy developed by the LINCC Frameworks team. It's meant to be a basic starting point that should allow you to run your own data through the model while maintaining reproducibility benefits of `Hyrax`.

## Environment

To run this notebook, you'll need to install the [hyrax_seldon](https://github.com/lincc-frameworks/hyrax_seldon) repo.

- create a new conda or venv environment
- download it with `git clone https://github.com/lincc-frameworks/hyrax_seldon.git`
- go into the directory `cd hyrax_seldon`
- install with `pip install -e .`
- create a kernel with `python -m ipykernel install --user --name={WHATEVER NAME YOU WANT}`

after that, you should be able to run everything SELDON related through `hyrax`.

In [1]:
import hyrax

## Files

You'll need 3 files to run SELDON through `hyrax`:

1. A file of light curve data
2. A pre-trained weights file
3. The `hparams.yaml` that goes along with the weights file.

Some examples listed below:

In [2]:
input_file = "./data/max_dp1_test_padded.parquet"

model_folder = "./seldon_model/"
weights_file = model_folder + "epoch=1086-val_loss=1.18.ckpt"
hparams_file = model_folder + "hparams.yaml"

## Input Columns
For the input format, you'll need the following columns with the same naming conventions:
- `SNID` : the unique id (if your data has a different id column, you can specify this later).
- `MJD` : `N` list of observation times for the light curve.
- `FLUXCAL` : `N` list of calibrated flux measurements.
- `FLUXCALERR` : `N` list of flux measurement errors.
- `BAND` : `N` list of obbservation bands.
- `PHOTFLAG` : `N` list of photometry bitmasks
- `RA` : RA position of the object.
- `Dec` : Dec position of the object.

Please note that for some iterations of `SELDON`, every entry must have at a minimum number `N` of observations to be processed without throwing an error.

## Setting Up `Hyrax`
To get started with `hyrax`, all we need to do is create a `hyrax` instance object.

In [3]:
hi = hyrax.Hyrax()

### Setting Up `data_request`
`Hyrax` manages and loads data through a `data_request`, which is set up per verb. The required parts of a `data_request` are:
```python
{
    "train" : { # the verb that you'll be working with (train, infer, test, umap etc)
        "blah" : { # user set name for the split, can be anything
            "dataset_class" : "<A HYRAX DATASET>", # the name or import string of a `hyrax` dataset
            "data_location" : "<PATH/TO/DATA>", # Path for data, optional for some dataset classes
            "primary_id_field" : "UNIQUE_ID", # the unique ID of the dataset
        }
    }
}
```

Here's a default setup using the example data above and running it through `hyrax`.

In [4]:
data_request = {
    "infer": {
        "data": {
            "dataset_class": "hyrax_seldon.datasets.elasticc_parquet.ElasticcParquetDataset",
            "data_location": input_file,
            "primary_id_field": "SNID",
        }
    }
}

hi.set_config("data_request", data_request, over_write=True)

[2026-11-19 16:41:22,983 hyrax.config_utils:INFO] Merging external default config from /u/mwest2/.conda/envs/max_seldon_test/lib/python3.12/site-packages/hyrax_seldon/default_config.toml


### Preparing the dataset (optional)

If you want to take a look at your data and ensure that everything has loaded properly before running inference, we can use the `prepare` verb. This will return a loaded version of the dataset that we can peak into.

In [5]:
ds = hi.prepare()

[2026-11-19 16:41:23,118 hyrax.verbs.prepare:INFO] Finished Prepare


We can access data within the `ds` dataset object by using the key of the verb that we loaded in the `data_request` above and take a peek at a specific index's values:

In [6]:
ds["infer"][0]["object_id"]

'2024aigs'

### Setting up the Model

In the `hyrax` config, we can set our config to point to the model we want to use. In this case, we'll go ahead and choose the hyraxified version of SELDON by setting `model.name` to the import string of our `HyraxSeldon` model:

In [7]:
hi.set_config("model.name", "hyrax_seldon.models.seldon.HyraxSeldon")

[2026-11-19 16:41:23,145 hyrax.config_utils:INFO] Merging external default config from /u/mwest2/.conda/envs/max_seldon_test/lib/python3.12/site-packages/hyrax_seldon/default_config.toml


We'll also need to point to the `model_weights_file` and for a SELDON specific setting (the "hparams.yaml" file that goes along with the weights) we can set the path as well:

In [8]:
hi.set_config("infer.model_weights_file", weights_file)

hi.set_config("model.HyraxSeldon.seldon_config", hparams_file)

[2026-11-19 16:41:23,178 hyrax.config_utils:INFO] Merging external default config from /u/mwest2/.conda/envs/max_seldon_test/lib/python3.12/site-packages/hyrax_seldon/default_config.toml
[2026-11-19 16:41:23,198 hyrax.config_utils:INFO] Merging external default config from /u/mwest2/.conda/envs/max_seldon_test/lib/python3.12/site-packages/hyrax_seldon/default_config.toml


A note on the `hyrax` config system: `hyrax` will automatically load in a number of default settings from a default config file, located both in the `hyrax` repo as well as whatever defaults are provided for the custom model. Using the config setting function above (or by creating a `.toml` config file of your own and loading it when you create a `hyrax` instance) you can make changes to way that `hyrax` operates, allowing you access to most switches and knobs that `pytorch` has, while providing you with a complete set of good defaults to start out with.

#### Config Links
- Read more about how the config system works [here](https://hyrax.readthedocs.io/en/stable/configuration_system.html)
- Look at the full list of base `hyrax` default configs [here](https://hyrax.readthedocs.io/en/stable/hyrax_default_config.html)
- For the default configs for the `hyrax_seldon` package, see [here](https://github.com/lincc-frameworks/hyrax_seldon/blob/main/src/hyrax_seldon/default_config.toml)

## Running Inference

Now that everything has been setup, we can go ahead and actually run inference! All we have to do is run the `infer` verb through our `hyrax` instance. The `infda` will return the result of the inference run so that you can look through them manually in this notebook, but by default they'll also be saved directly to disk in the results directory you provide (default in `general.results_dir` is "./results", but feel free to change it!).

In [9]:
infda = hi.infer()

[2026-11-19 16:41:29,613 hyrax.models.model_registry:INFO] Setting model's self.optimizer from config: torch.optim.SGD with arguments: {'lr': 0.01, 'momentum': 0.9}.
[2026-11-19 16:41:29,615 hyrax.models.model_registry:INFO] Setting model's self.criterion from config: torch.nn.CrossEntropyLoss with default arguments.
[2026-11-19 16:41:29,616 hyrax.models.model_registry:INFO] Setting model's self.scheduler from config: torch.optim.lr_scheduler.ExponentialLR
with arguments: {'gamma': 1}.
[2026-11-19 16:41:29,617 hyrax.verbs.infer:INFO] Inference model: HyraxSeldon
[2026-11-19 16:41:29,618 hyrax.verbs.infer:INFO] Inference dataset(s):
{'infer': Name: data (primary dataset)
  Dataset class: hyrax_seldon.datasets.elasticc_parquet.ElasticcParquetDataset
  Data location: /u/mwest2/data/max_dp1_test_padded.parquet
  Selected items: 11
  Primary ID field: SNID
  Requested fields: BAND, FLUXCAL, FLUXCALERR, MJD, PHOTFLAG, SNID
}
2026-11-19 16:41:29,635 ignite.distributed.auto.auto_dataloader INF

100%|##########| 1/1 [00:00<?, ?it/s]

[2026-11-19 16:41:32,797 hyrax.pytorch_ignite:INFO] Total evaluation time: 2.40[s]
[2026-11-19 16:41:32,799 hyrax.datasets.result_dataset:INFO] Optimizing Lance table after 1 batches
[2026-11-19 16:41:33,025 hyrax.datasets.result_dataset:INFO] Lance table optimization complete
[2026-11-19 16:41:33,028 hyrax.verbs.infer:INFO] Inference Complete.


Now by indexing through our inference results, we can get access to all the output of the `SELDON` model!

In [10]:
infda[0]

{'memory': array([ 8.58640075e-01,  8.22775185e-01,  4.37125266e-01,  4.81268287e-01,
        -3.62401605e-02,  6.58571482e-01,  2.32170179e-01,  1.59845638e+00,
         1.19470453e+00, -5.51272452e-01,  4.33572710e-01,  1.11143243e+00,
         2.99541414e-01,  7.82958865e-01,  9.13923621e-01, -1.05593467e+00,
         7.92267501e-01, -3.80456150e-01, -4.06288803e-01, -7.39652634e-01,
        -9.82020736e-01,  1.91587389e-01,  1.61288881e+00, -1.26345849e+00,
         2.13135064e-01, -6.00408792e-01, -5.70892930e-01,  8.71948242e-01,
        -4.33183551e-01, -7.64860749e-01,  9.67242122e-01,  4.98373091e-01,
        -3.39006543e-01,  3.45402300e-01, -2.67696142e-01,  2.54341304e-01,
         1.44606769e-01, -9.05663073e-02,  3.07896733e-01, -4.27740812e-01,
         3.96510899e-01, -7.56876349e-01, -1.29920971e+00,  4.20290053e-01,
         8.38121176e-02,  8.89332294e-01, -6.81470633e-02,  4.82255936e-01,
        -2.69081295e-01,  2.53364825e+00,  1.30778062e+00,  2.82248259e-01,
  